In [3]:
# Cell 01 | Initialize the Agentic RAG components
# Purpose: Reuse the verified persistent RAG and search tool setup from Lesson 13.
# Key points: This cell prepares the SQLite-backed retrieval, OpenAI client,
#             Python search function, and JSON tool schema for the agentic loop.
# Execution: Run from the notebooks directory. No LLM API request is made yet.

from dotenv import load_dotenv
from openai import OpenAI
from sqlitesearch import TextSearchIndex

from rag_helper import RAGBase


# Load the local OpenAI API key.
load_dotenv("../.env")

# Open the existing persistent FAQ index.
sqlite_index = TextSearchIndex(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"],
    db_path="faq.db",
)

# Create the OpenAI client.
client = OpenAI()

# Reuse the verified RAG abstraction and retrieval configuration.
assistant = RAGBase(
    index=sqlite_index,
    llm_client=client,
)


def search(query):
    """Search the LLM Zoomcamp FAQ using the verified project retrieval configuration."""
    return assistant.search(query)


# Describe search() in the tool format expected by the Responses API.
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the LLM Zoomcamp FAQ for information relevant to the user's question.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "The search query to use when retrieving relevant FAQ documents.",
            }
        },
        "required": ["query"],
        "additionalProperties": False,
    },
    "strict": True,
}


print("Documents available:", sqlite_index.count())
print("Model:", assistant.model)
print("Agentic RAG components ready.")

Documents available: 139
Model: gpt-5.6
Agentic RAG components ready.


In [4]:
# Cell 02 | Define the agent instructions
# Purpose: Define the behavioral rules that guide the agent during the loop.
# Key points: The model may search again when evidence is insufficient,
#             but should stop calling tools once it has enough information.
# Execution: This cell only defines instructions; it does not call the API.

agent_instructions = """
You are a helpful assistant for the LLM Zoomcamp course.

Use the search tool when you need factual information from the course FAQ.

Base course-specific answers on retrieved evidence rather than guessing.

If the retrieved information is insufficient, you may reformulate the search query
and call the search tool again.

When you have enough information to answer the user's question, provide the final
answer and stop calling tools.

If the available evidence does not contain enough information, say so clearly.
"""

print(agent_instructions)


You are a helpful assistant for the LLM Zoomcamp course.

Use the search tool when you need factual information from the course FAQ.

Base course-specific answers on retrieved evidence rather than guessing.

If the retrieved information is insufficient, you may reformulate the search query
and call the search tool again.

When you have enough information to answer the user's question, provide the final
answer and stop calling tools.

If the available evidence does not contain enough information, say so clearly.



In [5]:
# Cell 03 | Initialize the agent message history
# Purpose: Create the conversation state that will be preserved across loop iterations.
# Key points: The history begins with the user's question and will later include
#             model outputs, function calls, and function call results.
# Execution: This cell only initializes state; it does not call the API.

user_question = "Can I still join the LLM Zoomcamp course after it has started?"

history = [
    {
        "role": "user",
        "content": user_question,
    }
]

print("User question:")
print(user_question)

print("\nInitial history:")
print(history)

User question:
Can I still join the LLM Zoomcamp course after it has started?

Initial history:
[{'role': 'user', 'content': 'Can I still join the LLM Zoomcamp course after it has started?'}]


In [6]:
# Cell 04 | Execute one agent step
# Purpose: Let the model decide the next action based on instructions, tools,
#          and the current message history.
# Key points: Unlike Lesson 13, tool usage is not forced.
#             The model may request search() or return a final message.
# Execution: This cell makes one OpenAI API request.

response = client.responses.create(
    model=assistant.model,
    instructions=agent_instructions,
    input=history,
    tools=[search_tool],
)

print("Response output items:", len(response.output))

for index, item in enumerate(response.output, start=1):
    print(f"\nItem {index}")
    print("Type:", item.type)

    if item.type == "function_call":
        print("Function:", item.name)
        print("Arguments:", item.arguments)
        print("Call ID:", item.call_id)

    elif item.type == "message":
        print("Message:")
        print(response.output_text)

Response output items: 1

Item 1
Type: function_call
Function: search
Arguments: {"query":"Can I still join the LLM Zoomcamp course after it has started? late enrollment join after start"}
Call ID: call_VTByFBeWrJxQozvzXYyCGyze


In [7]:
# Cell 05 | Execute the requested tool and update agent history
# Purpose: Execute the function call requested by the model and return its result.
# Key points: Preserve the model output in history, parse JSON arguments,
#             execute the local search tool, and attach the result using call_id.
# Execution: This cell performs local retrieval only; it does not call the OpenAI API.

import json


# Preserve the complete model output in the agent history.
history.extend(response.output)

# Extract the function call requested by the model.
function_calls = [
    item for item in response.output
    if item.type == "function_call"
]

call = function_calls[0]

# Parse the model-generated JSON arguments.
arguments = json.loads(call.arguments)

print("Tool requested:", call.name)
print("Parsed arguments:", arguments)

# Execute the local tool.
if call.name == "search":
    tool_result = search(**arguments)
else:
    raise ValueError(f"Unknown tool requested: {call.name}")

print("\nRetrieved documents:", len(tool_result))

for index, document in enumerate(tool_result, start=1):
    print(f"\nResult {index}")
    print("Question:", document["question"])
    print("Section:", document["section"])

# Return the tool result to the matching function call.
tool_output = {
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": json.dumps(tool_result, ensure_ascii=False),
}

history.append(tool_output)

print("\nFunction call output added to history.")
print("History items:", len(history))

Tool requested: search
Parsed arguments: {'query': 'Can I still join the LLM Zoomcamp course after it has started? late enrollment join after start'}

Retrieved documents: 5

Result 1
Question: I just discovered the course. Can I still join?
Section: General Course-Related Questions

Result 2
Question: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
Section: General Course-Related Questions

Result 3
Question: Where can I track the LLM Zoomcamp syllabus, deadlines, homework, and progress?
Section: General Course-Related Questions

Result 4
Question: How should I start the course and follow the weekly workflow?
Section: General Course-Related Questions

Result 5
Question: Where is the LLM Zoomcamp Telegram channel?
Section: General Course-Related Questions

Function call output added to history.
History items: 3


In [8]:
# Cell 06 | Execute the second agent step
# Purpose: Let the model inspect the accumulated history and decide what to do next.
# Key points: The model can request another search or stop tool use and answer.
# Execution: This cell makes one OpenAI API request.

response_2 = client.responses.create(
    model=assistant.model,
    instructions=agent_instructions,
    input=history,
    tools=[search_tool],
)

print("Response output items:", len(response_2.output))

for index, item in enumerate(response_2.output, start=1):
    print(f"\nItem {index}")
    print("Type:", item.type)

    if item.type == "function_call":
        print("Function:", item.name)
        print("Arguments:", item.arguments)
        print("Call ID:", item.call_id)

    elif item.type == "message":
        print("Message:")
        print(response_2.output_text)

Response output items: 1

Item 1
Type: message
Message:
Yes, you can join LLM Zoomcamp after it has started. The videos and course materials remain available, so you can begin anytime.

If you want a certificate, make sure to submit the required project while submissions are still open. Check current deadlines on the [course management platform](https://courses.datatalks.club/llm-zoomcamp-2026/).


In [9]:
# Cell 07 | Finalize and inspect the manual agent trajectory
# Purpose: Preserve the second model response and inspect how the agent stopped.
# Key points: A final message with no new function call represents the stop condition
#             that will later control the automated agent loop.
# Execution: This cell only updates and inspects local state; no API call is made.

# Preserve the complete second model response in the agent history.
history.extend(response_2.output)

# Inspect the action types produced across the complete trajectory.
history_types = [
    item.type if hasattr(item, "type") else item.get("type", "unknown")
    for item in history
]

function_call_count = sum(
    item_type == "function_call"
    for item_type in history_types
)

function_output_count = sum(
    item_type == "function_call_output"
    for item_type in history_types
)

message_count = sum(
    item_type == "message"
    for item_type in history_types
)

print("History item types:")
print(history_types)

print("\nTrajectory summary:")
print("Function calls:", function_call_count)
print("Function outputs:", function_output_count)
print("Final messages:", message_count)

print("\nFinal answer:")
print(response_2.output_text)

History item types:
['unknown', 'function_call', 'function_call_output', 'message']

Trajectory summary:
Function calls: 1
Function outputs: 1
Final messages: 1

Final answer:
Yes, you can join LLM Zoomcamp after it has started. The videos and course materials remain available, so you can begin anytime.

If you want a certificate, make sure to submit the required project while submissions are still open. Check current deadlines on the [course management platform](https://courses.datatalks.club/llm-zoomcamp-2026/).


In [10]:
# Cell 08 | Build the reusable agentic loop
# Purpose: Convert the manually verified tool-calling trajectory into a reusable loop.
# Key points: The model decides whether to call a tool again or stop with a final answer.
#             A maximum iteration limit prevents uncontrolled looping.
# Execution: This cell defines run_agent(); it does not call the API yet.

import json


def run_agent(user_question, max_iterations=5):
    """Run an agentic RAG loop until the model answers or reaches the safety limit."""

    # Start every agent run with fresh task-local history.
    loop_history = [
        {
            "role": "user",
            "content": user_question,
        }
    ]

    for iteration in range(1, max_iterations + 1):
        print(f"\n=== Agent iteration {iteration} ===")

        # Ask the model to decide the next action.
        response = client.responses.create(
            model=assistant.model,
            instructions=agent_instructions,
            input=loop_history,
            tools=[search_tool],
        )

        # Preserve the complete model output for the next iteration.
        loop_history.extend(response.output)

        # Collect every function call returned in this response.
        function_calls = [
            item
            for item in response.output
            if item.type == "function_call"
        ]

        # No function call means the agent has stopped using tools.
        if not function_calls:
            print("Action: final answer")
            return {
                "answer": response.output_text,
                "history": loop_history,
                "iterations": iteration,
            }

        # Execute each requested tool call.
        for call in function_calls:
            print("Action: tool call")
            print("Tool:", call.name)
            print("Arguments:", call.arguments)

            arguments = json.loads(call.arguments)

            if call.name == "search":
                tool_result = search(**arguments)
            else:
                raise ValueError(f"Unknown tool requested: {call.name}")

            print("Retrieved documents:", len(tool_result))

            # Match the tool result to the model's call using call_id.
            loop_history.append(
                {
                    "type": "function_call_output",
                    "call_id": call.call_id,
                    "output": json.dumps(
                        tool_result,
                        ensure_ascii=False,
                    ),
                }
            )

    # Safety guard: the model kept requesting tools for too many iterations.
    raise RuntimeError(
        f"Agent reached the maximum of {max_iterations} iterations "
        "without producing a final answer."
    )


print("run_agent() ready.")

run_agent() ready.


In [11]:
# Cell 09 | Run and inspect the automated agent loop
# Purpose: Verify that the reusable loop can complete the same task automatically.
# Key points: The agent should decide when to search, consume the tool result,
#             and stop when it is ready to answer.
# Execution: This cell makes OpenAI API requests until the agent stops
#            or reaches the maximum iteration limit.

result = run_agent(
    "Can I still join the LLM Zoomcamp course after it has started?"
)

print("\n=== Agent completed ===")
print("Iterations:", result["iterations"])

print("\nFinal answer:")
print(result["answer"])


=== Agent iteration 1 ===
Action: tool call
Tool: search
Arguments: {"query":"Can I still join LLM Zoomcamp after the course has started? late enrollment joining late"}
Retrieved documents: 5

=== Agent iteration 2 ===
Action: final answer

=== Agent completed ===
Iterations: 2

Final answer:
Yes, you can still join after the LLM Zoomcamp has started.

You can begin learning immediately and submit homework while submission forms remain open. To receive a certificate, you must submit and pass the capstone project while project submissions are still being accepted.


In [12]:
# Cell 10 | Inspect the automated agent trajectory
# Purpose: Verify that run_agent() preserved the complete execution history.
# Key points: Inspect user input, function calls, tool outputs, and the final message.
# Execution: This cell only reads result["history"]; it makes no API requests.

agent_history = result["history"]


def get_item_type(item):
    """Return a readable type for both dictionaries and Responses API objects."""
    if isinstance(item, dict):
        if item.get("role") == "user":
            return "user"
        return item.get("type", "unknown")

    return getattr(item, "type", "unknown")


history_types = [
    get_item_type(item)
    for item in agent_history
]

print("History item types:")
print(history_types)

print("\nTrajectory details:")

for index, item in enumerate(agent_history, start=1):
    item_type = get_item_type(item)

    print(f"\nStep {index}")
    print("Type:", item_type)

    if item_type == "user":
        print("Content:", item["content"])

    elif item_type == "function_call":
        print("Function:", item.name)
        print("Arguments:", item.arguments)
        print("Call ID:", item.call_id)

    elif item_type == "function_call_output":
        tool_documents = json.loads(item["output"])
        print("Call ID:", item["call_id"])
        print("Retrieved documents:", len(tool_documents))

    elif item_type == "message":
        print("Final answer:")
        print(result["answer"])


print("\nTrajectory summary:")
print("Agent iterations:", result["iterations"])
print("Function calls:", history_types.count("function_call"))
print("Function outputs:", history_types.count("function_call_output"))
print("Final messages:", history_types.count("message"))

History item types:
['user', 'function_call', 'function_call_output', 'message']

Trajectory details:

Step 1
Type: user
Content: Can I still join the LLM Zoomcamp course after it has started?

Step 2
Type: function_call
Function: search
Arguments: {"query":"Can I still join LLM Zoomcamp after the course has started? late enrollment joining late"}
Call ID: call_JyzHMZ2Mxb06wLhFMtt4SZKd

Step 3
Type: function_call_output
Call ID: call_JyzHMZ2Mxb06wLhFMtt4SZKd
Retrieved documents: 5

Step 4
Type: message
Final answer:
Yes, you can still join after the LLM Zoomcamp has started.

You can begin learning immediately and submit homework while submission forms remain open. To receive a certificate, you must submit and pass the capstone project while project submissions are still being accepted.

Trajectory summary:
Agent iterations: 2
Function calls: 1
Function outputs: 1
Final messages: 1


In [13]:
# Cell 11 | Audit grounding evidence from the automated agent run
# Purpose: Inspect the retrieved FAQ evidence that supported the final answer.
# Key points: Reuse the existing function_call_output instead of running a new search.
#             Keep retrieval behavior unchanged during this audit.
# Execution: This cell only reads local agent history; no API request is made.

import json
import textwrap


# Find the retrieval result already stored in the agent trajectory.
tool_outputs = [
    item
    for item in agent_history
    if isinstance(item, dict)
    and item.get("type") == "function_call_output"
]

retrieved_documents = json.loads(tool_outputs[0]["output"])

print("Retrieved documents:", len(retrieved_documents))

for index, document in enumerate(retrieved_documents, start=1):
    answer_preview = textwrap.shorten(
        document["answer"].replace("\n", " "),
        width=700,
        placeholder=" ..."
    )

    print(f"\n=== Evidence {index} ===")
    print("Question:", document["question"])
    print("Section:", document["section"])
    print("Answer:")
    print(answer_preview)

Retrieved documents: 5

=== Evidence 1 ===
Question: I just discovered the course. Can I still join?
Section: General Course-Related Questions
Answer:
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

=== Evidence 2 ===
Question: Where can I track the LLM Zoomcamp syllabus, deadlines, homework, and progress?
Section: General Course-Related Questions
Answer:
Use the [LLM Zoomcamp course management platform](https://courses.datatalks.club/llm-zoomcamp-2026/). It contains the current cohort structure, homework, deadlines, and progress tracking. The process is the same as in other DataTalks.Club Zoomcamps.

=== Evidence 3 ===
Question: Where is the LLM Zoomcamp Telegram channel?
Section: General Course-Related Questions
Answer:
The Telegram channel is [https://t.me/llm_zoomcamp](https://t.me/llm_zoomcamp). Use it for announcements. For technical discussion and questions, use the course Slack channel.

=== Evidence 4 ===